# Environment Sanity Check
Validate Gymnasium MountainCar environment behavior, spaces, rewards, terminations, and seed reproducibility.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import gymnasium as gym
import numpy as np
from src.envs.mountain_car_discrete import make_discrete_env, get_discrete_env_spec
from src.envs.mountain_car_continuous import make_continuous_env, get_continuous_env_spec
from src.utils.seeding import seed_everything, seed_env

In [ ]:
seed_everything(42)
print(get_discrete_env_spec())
print(get_continuous_env_spec())

In [ ]:
def random_rollout(env, episodes=3):
    all_returns = []
    done_types = []
    for _ in range(episodes):
        obs, info = env.reset()
        done = False
        total_reward = 0.0
        steps = 0
        while not done:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += float(reward)
            done = bool(terminated or truncated)
            steps += 1
        all_returns.append(total_reward)
        done_types.append((terminated, truncated, steps))
    return all_returns, done_types

disc_env = make_discrete_env()
cont_env = make_continuous_env()

disc_returns, disc_done = random_rollout(disc_env, episodes=5)
cont_returns, cont_done = random_rollout(cont_env, episodes=5)

print('Discrete returns:', disc_returns)
print('Discrete done flags:', disc_done)
print('Continuous returns:', cont_returns)
print('Continuous done flags:', cont_done)

disc_env.close()
cont_env.close()

In [ ]:
# Reproducibility check: same seed should produce same initial obs and sampled action sequence.
seed = 123
env_a = make_discrete_env()
env_b = make_discrete_env()
seed_env(env_a, seed)
seed_env(env_b, seed)
obs_a, _ = env_a.reset(seed=seed)
obs_b, _ = env_b.reset(seed=seed)
print('Same initial obs:', np.allclose(obs_a, obs_b))

acts_a = [env_a.action_space.sample() for _ in range(10)]
acts_b = [env_b.action_space.sample() for _ in range(10)]
print('Same sampled actions:', acts_a == acts_b)

env_a.close()
env_b.close()

In [ ]:
# Optional frame extraction in rgb_array mode.
frame_env = make_discrete_env(render_mode='rgb_array')
obs, _ = frame_env.reset(seed=42)
frames = []
for _ in range(25):
    action = frame_env.action_space.sample()
    obs, reward, terminated, truncated, _ = frame_env.step(action)
    frame = frame_env.render()
    if frame is not None:
        frames.append(frame)
    if terminated or truncated:
        break

frame_env.close()
print('Captured frames:', len(frames))